# Практична робота №4. Чищення набору даних

**Студент:** Панасюк Дмитро  
**Номер за списком:** 15  
**Варіант:** №5 — Дніпро  
**Тема:** Пропуски, дублікати, неправильні типи, неузгоджені категорії та викиди

**Дані варіанта:** базова ціна товару — **810 грн**, базова кількість — **3 шт.**  
**Товар:** настільна лампа. Усі дані про замовлення навчальні, створені спеціально для цієї практичної роботи.

**Мета:** сформувати набір із п’ятьма видами дефектів, послідовно знайти та усунути їх за допомогою `pandas` і `NumPy`.

## Підготовка середовища
Імпортуємо необхідні бібліотеки. Ноутбук можна виконувати в Google Colab або локальному Jupyter.

In [1]:
import numpy as np
import pandas as pd

print("Версія pandas:", pd.__version__)
print("Версія NumPy:", np.__version__)

Версія pandas: 2.2.3
Версія NumPy: 2.3.5


## Завдання 1. Побудова «брудного» набору даних

Створюємо рівно **8 рядків замовлень** для одного товару в Дніпрі. Навмисно вводимо такі дефекти:

1. Два пропуски в `кількість` (`NaN`) — замовлення 102 і 105.
2. Один повний дублікат — два абсолютно однакові рядки із `id_замовлення = 106`.
3. Дві ціни записані текстом із суфіксом ` грн` — `810 грн` і `830 грн`.
4. Три варіанти назви міста — `Дніпро`, ` Дніпро `, `ДНІПРО`.
5. Одна аномальна ціна — **8100 грн**, тобто у 10 разів більше базової (810 грн).

Значення інших цін близькі до базової ціни й моделюють невеликі знижки/відмінності у вартості.

In [2]:
dirty = pd.DataFrame([
    {"id_замовлення": 101, "дата": "2026-09-01", "товар": "Настільна лампа", "місто": "Дніпро",   "ціна": "810 грн", "кількість": 3},
    {"id_замовлення": 102, "дата": "2026-09-02", "товар": "Настільна лампа", "місто": " Дніпро ", "ціна": 790,       "кількість": np.nan},
    {"id_замовлення": 103, "дата": "2026-09-03", "товар": "Настільна лампа", "місто": "ДНІПРО",  "ціна": "830 грн", "кількість": 3},
    {"id_замовлення": 104, "дата": "2026-09-04", "товар": "Настільна лампа", "місто": "Дніпро",   "ціна": 820,       "кількість": 2},
    {"id_замовлення": 105, "дата": "2026-09-05", "товар": "Настільна лампа", "місто": " Дніпро ", "ціна": 800,       "кількість": np.nan},
    {"id_замовлення": 106, "дата": "2026-09-06", "товар": "Настільна лампа", "місто": "ДНІПРО",  "ціна": 815,       "кількість": 8},
    {"id_замовлення": 106, "дата": "2026-09-06", "товар": "Настільна лампа", "місто": "ДНІПРО",  "ціна": 815,       "кількість": 8},
    {"id_замовлення": 107, "дата": "2026-09-07", "товар": "Настільна лампа", "місто": "Дніпро",   "ціна": 8100,      "кількість": 3},
])

print("Розмір вихідної таблиці:", dirty.shape)
print("Типи стовпців:")
print(dirty.dtypes)
display(dirty)

Розмір вихідної таблиці: (8, 6)
Типи стовпців:
id_замовлення      int64
дата              object
товар             object
місто             object
ціна              object
кількість        float64
dtype: object


,id_замовлення,дата,товар,місто,ціна,кількість
0,101,2026-09-01,Настільна лампа,Дніпро,810 грн,3.0
1,102,2026-09-02,Настільна лампа,Дніпро,790,NaN
2,103,2026-09-03,Настільна лампа,ДНІПРО,830 грн,3.0
3,104,2026-09-04,Настільна лампа,Дніпро,820,2.0
4,105,2026-09-05,Настільна лампа,Дніпро,800,NaN
5,106,2026-09-06,Настільна лампа,ДНІПРО,815,8.0
6,106,2026-09-06,Настільна лампа,ДНІПРО,815,8.0
7,107,2026-09-07,Настільна лампа,Дніпро,8100,3.0


### Перевірка наявності всіх п’яти дефектів

Ця перевірка підтверджує, що початковий набір справді відповідає вимогам завдання.

In [3]:
checks = {
    "8 рядків": len(dirty) == 8,
    "2 пропуски кількості": dirty["кількість"].isna().sum() == 2,
    "1 повний дублікат": dirty.duplicated().sum() == 1,
    "2 текстові ціни з ' грн'": dirty["ціна"].astype(str).str.endswith(" грн").sum() >= 2,
    "3 різні варіанти міста": dirty["місто"].nunique() == 3,
    "Ціна 8100 грн = 10 × 810 грн": (dirty["ціна"] == 8100).sum() == 1,
}
for name, passed in checks.items():
    print(f"{'✓' if passed else '✗'} {name}: {passed}")
assert all(checks.values()), "Не всі дефекти присутні у вихідному наборі!"

✓ 8 рядків: True
✓ 2 пропуски кількості: True
✓ 1 повний дублікат: True
✓ 2 текстові ціни з ' грн': True
✓ 3 різні варіанти міста: True
✓ Ціна 8100 грн = 10 × 810 грн: True


## Завдання 2. Пропуски

1. Підраховуємо `NaN` у **кожному** стовпці.
2. Заповнюємо два пропуски в `кількість` медіаною наявних значень.

**Чому медіана?** Усього 8 рядків, тому окреме велике замовлення на 8 штук може помітно змінити середнє. Медіана стійкіша до таких крайніх значень і краще передає типову кількість (3 шт.).

In [4]:
print("Пропуски до очищення:")
print(dirty.isna().sum())

median_qty = dirty["кількість"].median()
mean_qty = dirty["кількість"].mean()
print(f"\nМедіана кількості: {median_qty:g}")
print(f"Середнє кількості: {mean_qty:.2f}")

orders = dirty.copy()
orders["кількість"] = orders["кількість"].fillna(median_qty).astype(int)

print("\nПропуски після заповнення:")
print(orders.isna().sum())
display(orders[["id_замовлення", "кількість"]])
assert orders.isna().sum().sum() == 0

Пропуски до очищення:
id_замовлення    0
дата             0
товар            0
місто            0
ціна             0
кількість        2
dtype: int64

Медіана кількості: 3
Середнє кількості: 4.50

Пропуски після заповнення:
id_замовлення    0
дата             0
товар            0
місто            0
ціна             0
кількість        0
dtype: int64


,id_замовлення,кількість
0,101,3
1,102,3
2,103,3
3,104,2
4,105,3
5,106,8
6,106,8
7,107,3


## Завдання 3. Дублікати

- `duplicated()` без `subset` шукає повний збіг **усіх стовпців**.
- `duplicated(subset=["id_замовлення"])` перевіряє повторення ідентифікатора замовлення. У нашому наборі кожне реальне замовлення має унікальний ID.
- Після видалення одного дубліката має залишитися **7 рядків**.

In [5]:
full_duplicates = orders.duplicated()
key_duplicates = orders.duplicated(subset=["id_замовлення"])

print("Повних дублікатів:", full_duplicates.sum())
print("Дублікатів за ID замовлення:", key_duplicates.sum())
print("\nПовторені рядки (за повним збігом):")
display(orders.loc[full_duplicates])
print("Повторені ID:")
display(orders.loc[key_duplicates, ["id_замовлення", "дата", "товар"]])

orders = orders.drop_duplicates(subset=["id_замовлення"], keep="first").reset_index(drop=True)
print("Рядків після усунення дубліката:", len(orders))
assert len(orders) == 7
assert not orders.duplicated().any()
assert not orders.duplicated(subset=["id_замовлення"]).any()

Повних дублікатів: 1
Дублікатів за ID замовлення: 1

Повторені рядки (за повним збігом):


,id_замовлення,дата,товар,місто,ціна,кількість
6,106,2026-09-06,Настільна лампа,ДНІПРО,815,8


Повторені ID:


,id_замовлення,дата,товар
6,106,2026-09-06,Настільна лампа


Рядків після усунення дубліката: 7


## Завдання 4. Типи даних і узгодження категорій

1. Прибираємо суфікс ` грн`, потім перетворюємо `ціна` на `float`.
2. Нормалізуємо назви міста: відсікаємо крайні пробіли, переводимо літери в нижній регістр і замінюємо результат на канонічний запис **«Дніпро»**.
3. Перевіряємо тип ціни і список унікальних назв міста.

In [6]:
print("Тип ціни до очищення:", orders["ціна"].dtype)
print("Варіанти міста до очищення:", orders["місто"].unique())

orders["ціна"] = (
    orders["ціна"]
    .astype(str)
    .str.replace(" грн", "", regex=False)
    .astype(float)
)
orders["місто"] = (
    orders["місто"]
    .str.strip()
    .str.lower()
    .replace({"дніпро": "Дніпро"})
)

print("\nТип ціни після очищення:", orders["ціна"].dtype)
print("Унікальні назви міста після очищення:", orders["місто"].unique())
display(orders)

assert pd.api.types.is_float_dtype(orders["ціна"])
assert orders["місто"].unique().tolist() == ["Дніпро"]

Тип ціни до очищення: object
Варіанти міста до очищення: ['Дніпро' ' Дніпро ' 'ДНІПРО']

Тип ціни після очищення: float64
Унікальні назви міста після очищення: ['Дніпро']


,id_замовлення,дата,товар,місто,ціна,кількість
0,101,2026-09-01,Настільна лампа,Дніпро,810.0,3
1,102,2026-09-02,Настільна лампа,Дніпро,790.0,3
2,103,2026-09-03,Настільна лампа,Дніпро,830.0,3
3,104,2026-09-04,Настільна лампа,Дніпро,820.0,2
4,105,2026-09-05,Настільна лампа,Дніпро,800.0,3
5,106,2026-09-06,Настільна лампа,Дніпро,815.0,8
6,107,2026-09-07,Настільна лампа,Дніпро,8100.0,3


## Завдання 5. Виявлення та обробка викиду методом IQR

Обчислюємо квартили `Q1` і `Q3`, міжквартильний розмах `IQR = Q3 − Q1` та межі:

- **Нижня:** `Q1 − 1.5 × IQR`.
- **Верхня:** `Q3 + 1.5 × IQR`.

Вихідну аномальну ціну **8100 грн** перевіряємо після приведення стовпця до числового типу. Зверніть увагу: «за межами IQR» означає *підозріле значення*, а не автоматично помилку.

In [7]:
q1 = orders["ціна"].quantile(0.25)
q3 = orders["ціна"].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = orders[(orders["ціна"] < lower_bound) | (orders["ціна"] > upper_bound)]

print(f"Q1 = {q1:.2f} грн")
print(f"Q3 = {q3:.2f} грн")
print(f"IQR = {iqr:.2f} грн")
print(f"Нижня межа = {lower_bound:.2f} грн")
print(f"Верхня межа = {upper_bound:.2f} грн")
print(f"Кількість підозрілих цін: {len(outliers)}")
display(outliers[["id_замовлення", "місто", "ціна", "кількість"]])
assert outliers["id_замовлення"].tolist() == [107]

Q1 = 805.00 грн
Q3 = 825.00 грн
IQR = 20.00 грн
Нижня межа = 775.00 грн
Верхня межа = 855.00 грн
Кількість підозрілих цін: 1


,id_замовлення,місто,ціна,кількість
6,107,Дніпро,8100.0,3


### Рішення щодо ціни 8100 грн

Я обираю трактувати цю ціну як **навмисно змодельовану помилку введення**: у нас інтернет-магазин одного товару з базовою ціною **810 грн**, а для замовлення №107 помилково записано **8100 грн** (зайвий нуль). Значення ціни за одиницю виправляємо на **810 грн**, **не видаляючи** реальне за задумом замовлення.

У справжньому магазині перед виправленням ціни потрібно звірити її з каталогом або первинними документами; сама по собі межа IQR не доводить помилку. Тут правильна ціна відома, оскільки набір сконструйований нами.

In [8]:
orders.loc[orders["id_замовлення"] == 107, "ціна"] = 810.0

# Повторно застосовуємо той самий критерій до виправленого набору.
q1_after = orders["ціна"].quantile(0.25)
q3_after = orders["ціна"].quantile(0.75)
iqr_after = q3_after - q1_after
lower_after = q1_after - 1.5 * iqr_after
upper_after = q3_after + 1.5 * iqr_after
outliers_after = orders[(orders["ціна"] < lower_after) | (orders["ціна"] > upper_after)]

print("Ціна замовлення №107 після виправлення:",
      orders.loc[orders["id_замовлення"] == 107, "ціна"].iloc[0], "грн")
print(f"Нові межі IQR: [{lower_after:.2f}; {upper_after:.2f}] грн")
print("Підозрілих цін після виправлення:", len(outliers_after))
assert outliers_after.empty

Ціна замовлення №107 після виправлення: 810.0 грн
Нові межі IQR: [786.25; 836.25] грн
Підозрілих цін після виправлення: 0


## Фінальна перевірка очищеного набору

Переконуємося, що лишилось 7 унікальних замовлень, немає пропусків, повних дублікатів, текстових цін, різних написань назви міста чи аномальної ціни.

In [9]:
print("Очищений DataFrame:")
display(orders)

summary = {
    "7 рядків": len(orders) == 7,
    "немає пропусків": orders.isna().sum().sum() == 0,
    "немає повних дублікатів": orders.duplicated().sum() == 0,
    "унікальні ID замовлень": orders["id_замовлення"].is_unique,
    "ціна має тип float": pd.api.types.is_float_dtype(orders["ціна"]),
    "місто у єдиному записі": orders["місто"].unique().tolist() == ["Дніпро"],
    "викид за IQR виправлено": len(outliers_after) == 0,
}
for name, passed in summary.items():
    print(f"{'✓' if passed else '✗'} {name}")
assert all(summary.values())

Очищений DataFrame:


,id_замовлення,дата,товар,місто,ціна,кількість
0,101,2026-09-01,Настільна лампа,Дніпро,810.0,3
1,102,2026-09-02,Настільна лампа,Дніпро,790.0,3
2,103,2026-09-03,Настільна лампа,Дніпро,830.0,3
3,104,2026-09-04,Настільна лампа,Дніпро,820.0,2
4,105,2026-09-05,Настільна лампа,Дніпро,800.0,3
5,106,2026-09-06,Настільна лампа,Дніпро,815.0,8
6,107,2026-09-07,Настільна лампа,Дніпро,810.0,3


✓ 7 рядків
✓ немає пропусків
✓ немає повних дублікатів
✓ унікальні ID замовлень
✓ ціна має тип float
✓ місто у єдиному записі
✓ викид за IQR виправлено


## Письмові відповіді на контрольні питання

**1. Чому заповнення пропусків середнім, а не медіаною, може згодом спотворити стандартне відхилення?**

Середнє чутливе до крайніх значень. У нашому прикладі замовлення на 8 штук підвищують середню кількість до **4,50**, хоча типове значення за медіаною — **3**. Заповнення пропусків середнім додало б у таблицю значення 4,50, яких реально не спостерігали, і змінило б розподіл, а отже, і подальшу оцінку стандартного відхилення. Заповнення медіаною також впливає на розкид, але менше залежить від крайніх спостережень. Жодне таке заповнення не гарантує точного збереження справжньої дисперсії.

**2. Чим відрізняються `duplicated()` без `subset` і з `subset`? Чому результат може відрізнятися?**

`duplicated()` без аргументів порівнює **всі стовпці** й позначає повтор лише за повного збігу рядків. `duplicated(subset=["id_замовлення"])` порівнює **тільки ідентифікатори замовлень**: рядки з однаковим ID будуть позначені як дублікати, навіть якщо відрізняються, наприклад, ціною або назвою міста. У нашому наборі обидва підходи знаходять один дублікат, але загалом їхні результати можуть різнитися. Набір полів для `subset` потрібно обирати відповідно до того, що робить запис унікальним.

**3. Чому автоматично видаляти всі значення за межами IQR — не завжди правильно?**

IQR виявляє **статистично незвичні**, а не обов’язково помилкові значення. Наприклад, велика ціна може бути реальною у разі спеціального товару або іншого типу операції, а велика кількість товару може означати оптову покупку. Автоматичне видалення таких рядків призвело б до втрати справжніх даних і можливого спотворення результатів аналізу. У цій навчальній таблиці значення 8100 грн визнано помилковим не лише через IQR, а й тому, що ми самі спеціально ввели зайвий нуль відносно відомої базової ціни 810 грн.

---

**Висновок.** Побудовано набір із 8 замовлень і всіма 5 заданими дефектами; два пропуски заповнено медіаною, один повтор видалено, ціни перетворено на `float`, назву міста стандартизовано, а аномальну ціну свідомо виправлено. В результаті одержано 7 коректних записів.